Benchmark TrunCat & TrunKitten vs NMDetective-BCompares our two CatBoost escape models against a locally-reimplemented NMDetective-B decision tree (Lindeboom et al., Nat Genet 2019, Fig. 1c) on the same TOPMed held-out variants.**Framing:** TrunCat/TrunKitten *refine* NMDetective-B's coarse 5-bin output,not "beat" it. NMDetective-B was trained on a different signal (cancertranscriptomics regression target, not ASE-derived binary labels), so thefairest read of these results is "how much does our model add over thefield-standard rule-based predictor on this cohort."**Outputs**- `outputs/benchmark_nmdetectiveB/performance_summary.tsv` — main AUC table- `outputs/benchmark_nmdetectiveB/within_bin_*.tsv` — stratified AUCs- `outputs/benchmark_nmdetectiveB/binary_metrics.tsv` — sens/spec/PPV/etc.- `outputs/benchmark_nmdetectiveB/figs/` — ROC, PR, stratification plots

## 1. Configuration All file paths and column names are here.

In [ ]:
from pathlib import Path
import sys

# --- Paths -----------------------------------------------------------------
BASE_DIR = Path("../../..").resolve()  # repo root; notebook lives at Model/Benchmarking/NMDetective-B/

# OOF/CV prediction files with IDs added
TRUNCAT_OOF_PATH = BASE_DIR / "Model" / "TrunCat" / "results" / "cv_predictions_with_ids.csv"

TRUNKITTEN_OOF_PATH = BASE_DIR / "Model" / "TrunKitten" / "results" / "cv_predictions_with_ids.csv"

# NMDetective-B rule output
NMDETECTIVE_B_PATH = BASE_DIR / "Features" / "nmdetective" / "outputs" / \
                     "topmed_nmdetectiveB_rules.tsv"

# Output directory for benchmark
OUTPUT_DIR = BASE_DIR / "Model" / "TrunCat" / "results" / "benchmark_nmdetectiveB"
FIG_DIR = OUTPUT_DIR / "figs"

# --- Column names -----------------------------------------------------------
# Stable variant-level ID shared across TrunCat, TrunKitten, and NMDetective-B.
# This is used for all merges.
VARIANT_ID_COL = "key"

# In OOF prediction tables
LABEL_COL_OOF = "y_true"          # 1 = escape, 0 = NMD-sensitive
TRUNCAT_SCORE_COL = "oof_prob"    # P(escape)
TRUNKITTEN_SCORE_COL = "oof_prob" # P(escape)

# In NMDetective-B output table
NMDETECTIVE_RAW_COL = "NMDetective_B_score"   # higher = more NMD-sensitive
NMDETECTIVE_RULE_COL = "NMDetective_B_rule"
NMDETECTIVE_CLASS_COL = "NMDetective_B_class"

# --- Bootstrap settings ----------------------------------------------------
N_BOOT = 1000
BOOT_SEED = 42

# --- Bin order for plots ----------------------------------------------------
RULE_ORDER = [
    "last_exon",
    "start_proximal",
    "50nt_rule",
    "long_exon",
    "trigger_NMD",
]

# --- Add benchmark utils to path -------------------------------------------
sys.path.insert(0, str(Path.cwd()))
import benchmark_utils as bu

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"TRUNCAT_OOF_PATH: {TRUNCAT_OOF_PATH}")
print(f"TRUNKITTEN_OOF_PATH: {TRUNKITTEN_OOF_PATH}")
print(f"NMDETECTIVE_B_PATH: {NMDETECTIVE_B_PATH}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## 2. Loads the three input tables and merges on `variant_id`.

In [ ]:
import pandas as pd
import numpy as np

def read_table(path):
    path = Path(path)
    
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    
    if path.suffix.lower() == ".csv":
        sep = ","
    elif path.suffix.lower() in [".tsv", ".txt"]:
        sep = "\t"
    else:
        raise ValueError(f"Unknown file type for {path}. Expected .csv, .tsv, or .txt")
    
    return pd.read_csv(path, sep=sep, low_memory=False)

truncat    = read_table(TRUNCAT_OOF_PATH)
trunkitten = read_table(TRUNKITTEN_OOF_PATH)
nmd_b      = read_table(NMDETECTIVE_B_PATH)

print(f"TrunCat OOF:    {len(truncat):>6,} rows, {len(truncat.columns)} cols")
print(f"TrunKitten OOF: {len(trunkitten):>6,} rows, {len(trunkitten.columns)} cols")
print(f"NMDetective-B:  {len(nmd_b):>6,} rows, {len(nmd_b.columns)} cols")

print("\nTrunCat columns:   ", list(truncat.columns)[:20])
print("TrunKitten columns:", list(trunkitten.columns)[:20])
print("NMDetective cols:  ", list(nmd_b.columns)[:20])

In [ ]:
# Merge by variant_id only
required = {
    "truncat": [VARIANT_ID_COL, LABEL_COL_OOF, TRUNCAT_SCORE_COL],
    "trunkitten": [VARIANT_ID_COL, TRUNKITTEN_SCORE_COL],
    "nmd_b": [VARIANT_ID_COL, NMDETECTIVE_RAW_COL, NMDETECTIVE_RULE_COL, NMDETECTIVE_CLASS_COL],
}

for name, cols in required.items():
    missing = [c for c in cols if c not in globals()[name].columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

tc = truncat[[VARIANT_ID_COL, LABEL_COL_OOF, TRUNCAT_SCORE_COL]].rename(
    columns={LABEL_COL_OOF: "y_true", TRUNCAT_SCORE_COL: "score_truncat"}
)

tk = trunkitten[[VARIANT_ID_COL, TRUNKITTEN_SCORE_COL]].rename(
    columns={TRUNKITTEN_SCORE_COL: "score_trunkitten"}
)

nb = nmd_b[[VARIANT_ID_COL, NMDETECTIVE_RAW_COL, NMDETECTIVE_RULE_COL, NMDETECTIVE_CLASS_COL]].rename(
    columns={
        NMDETECTIVE_RAW_COL: "nmd_b_raw",
        NMDETECTIVE_RULE_COL: "nmd_b_rule",
        NMDETECTIVE_CLASS_COL: "nmd_b_class",
    }
)

df = (
    tc.merge(tk, on=VARIANT_ID_COL, how="inner", validate="one_to_one")
      .merge(nb, on=VARIANT_ID_COL, how="inner", validate="one_to_one")
)

# Direction-flipped NMDetective-B score:
# higher = more escape, matching TrunCat / TrunKitten direction
df["score_nmd_b"] = 1.0 - df["nmd_b_raw"]

print(f"[merge] {len(df):,} variants after inner join by {VARIANT_ID_COL}")
df.head()

## 3. Sanity checks
Catch direction errors and parsing issues before computing metrics.

In [ ]:

print("=" * 60)
print("SANITY CHECKS")
print("=" * 60)

# 3a. Missing values
print("\n[3a] Missing values per column:")
print(df.isna().sum().to_string())

# 3b. Label distribution
print("\n[3b] Label distribution:")
print(df["y_true"].value_counts().to_string())
print(f"  Escape prevalence: {df['y_true'].mean():.3f}")

# 3c. NMDetective-B rule and class breakdown
print("\n[3c] NMDetective-B rule counts:")
print(df["nmd_b_rule"].value_counts().to_string())
print("\n[3c] NMDetective-B class counts:")
print(df["nmd_b_class"].value_counts().to_string())

# 3d. Direction sanity: last_exon (raw=0.00) should map to escape score 1.00
last_exon_rows = df[df["nmd_b_rule"] == "last_exon"]
if len(last_exon_rows) > 0:
    raw_vals = last_exon_rows["nmd_b_raw"].unique()
    flipped_vals = last_exon_rows["score_nmd_b"].unique()
    print(f"\n[3d] last_exon: raw values={raw_vals}, flipped={flipped_vals}")
    assert np.allclose(flipped_vals, 1.0), \
        "Direction flip is wrong: last_exon should map to score_nmd_b == 1.0"

# 3e. All scores should be 'higher = more escape'. For NMDetective-B, this
# means last_exon variants should have HIGHER score_nmd_b than trigger_NMD.
mean_le = df.loc[df["nmd_b_rule"] == "last_exon",   "score_nmd_b"].mean()
mean_tn = df.loc[df["nmd_b_rule"] == "trigger_NMD", "score_nmd_b"].mean()
print(f"\n[3e] mean score_nmd_b: last_exon={mean_le:.3f}, trigger_NMD={mean_tn:.3f}")
assert mean_le > mean_tn, "Direction error: last_exon should have higher escape score"

# 3f. TrunCat and TrunKitten should rank escape > sensitive on average
for sc in ["score_truncat", "score_trunkitten"]:
    by_label = df.groupby("y_true")[sc].mean()
    print(f"[3f] {sc}: mean for sensitive={by_label[0]:.3f}, escape={by_label[1]:.3f}")
    if by_label[1] <= by_label[0]:
        print(f"     WARNING: {sc} is not higher for escape — direction may be flipped")

# 3g. Score ranges
print("\n[3g] Score ranges:")
for sc in ["score_truncat", "score_trunkitten", "score_nmd_b"]:
    print(f"  {sc:18s}  min={df[sc].min():.4f}  max={df[sc].max():.4f}  "
          f"distinct_values={df[sc].nunique()}")

print("\nAll sanity checks passed ✓")

## 4. Primary benchmark — ROC-AUC and PR-AUC with bootstrap CIs 
NMDetective-B's 5 distinct values produce a step-functionROC curve, but AUC is well-defined (sklearn handles tied predictions correctly).
Same bootstrap seed is used for all three models, so the resampled variantindices are matched across models — this makes the CIs directly comparable.

In [ ]:

y_true = df["y_true"].values

print("Computing metrics with bootstrap CIs...")
metrics = {}
for name, score_col in [
    ("TrunCat",       "score_truncat"),
    ("TrunKitten",    "score_trunkitten"),
    ("NMDetective-B", "score_nmd_b"),
]:
    metrics[name] = bu.continuous_metrics(
        y_true, df[score_col].values, n_boot=N_BOOT, seed=BOOT_SEED,
    )
    m = metrics[name]
    print(f"  {name:14s}  ROC-AUC={m['roc_auc']:.3f} "
          f"({m['roc_auc_lo']:.3f}-{m['roc_auc_hi']:.3f})  "
          f"PR-AUC={m['pr_auc']:.3f} "
          f"({m['pr_auc_lo']:.3f}-{m['pr_auc_hi']:.3f})")

summary = pd.DataFrame([
    {"model": k, **v} for k, v in metrics.items()
])
summary.to_csv(OUTPUT_DIR / "performance_summary.tsv", sep="\t", index=False)
print(f"\n[write] {OUTPUT_DIR / 'performance_summary.tsv'}")
summary

## 5. Main figures — ROC and PR overlays produces PNG + PDF. 
NMDetective-B is rendered as a stepcurve to make the discrete-score nature visually explicit.

In [ ]:

# Format legend strings with AUC and CI
def legend_str(name, kind):
    m = metrics[name]
    if kind == "roc":
        return (f"{name}: AUC = {m['roc_auc']:.3f} "
                f"({m['roc_auc_lo']:.3f}–{m['roc_auc_hi']:.3f})")
    return (f"{name}: PR-AUC = {m['pr_auc']:.3f} "
            f"({m['pr_auc_lo']:.3f}–{m['pr_auc_hi']:.3f})")

roc_inputs = {
    "TrunCat":       {"y_true": y_true, "y_score": df["score_truncat"].values,
                      "legend": legend_str("TrunCat", "roc")},
    "TrunKitten":    {"y_true": y_true, "y_score": df["score_trunkitten"].values,
                      "legend": legend_str("TrunKitten", "roc")},
    "NMDetective-B": {"y_true": y_true, "y_score": df["score_nmd_b"].values,
                      "legend": legend_str("NMDetective-B", "roc")},
}
pr_inputs = {
    name: {"y_true": d["y_true"], "y_score": d["y_score"],
           "legend": legend_str(name, "pr")}
    for name, d in roc_inputs.items()
}

fig_roc, _ = bu.plot_roc_overlay(
    roc_inputs,
    title="ROC: NMD escape prediction on TOPMed held-out set",
    save_path=FIG_DIR / "roc_overlay.png",
)
fig_pr, _ = bu.plot_pr_overlay(
    pr_inputs,
    title="Precision-recall: NMD escape prediction on TOPMed held-out set",
    save_path=FIG_DIR / "pr_overlay.png",
)
print(f"[write] {FIG_DIR / 'roc_overlay.png'} (+ .pdf)")
print(f"[write] {FIG_DIR / 'pr_overlay.png'} (+ .pdf)")

## 6. Secondary benchmark — binary classification metrics
Three threshold scenarios for TrunCat/TrunKitten:1. 
**Default 0.5** — what a naive user would do2. 
**Youden's J** — optimal under equal weighting of sens/spec3. 
**Matched specificity** — TrunCat/TrunKitten thresholded to match   NMDetective-B's specificity, so sensitivity comparison is apples-to-apples
NMDetective-B uses Lindeboom's published cutoff: raw score > 0.52 → sensitive.
Equivalently in flipped scoring: `score_nmd_b < 0.48` → sensitive,`score_nmd_b >= 0.48` → escape-like (collapses escape + intermediate, matchingLindeboom's "efficient NMD vs not" framing).

In [ ]:

# NMDetective-B binary classification at the published cutoff
NMD_B_ESCAPE_THRESHOLD = 1.0 - 0.52   # = 0.48 in flipped space
nmd_b_metrics = bu.binary_metrics(
    y_true, df["score_nmd_b"].values, threshold=NMD_B_ESCAPE_THRESHOLD,
)
print(f"NMDetective-B (raw > 0.52 → sensitive):")
print(f"  threshold (flipped space) = {NMD_B_ESCAPE_THRESHOLD:.3f}")
print(f"  sensitivity = {nmd_b_metrics['sensitivity']:.3f}")
print(f"  specificity = {nmd_b_metrics['specificity']:.3f}")
print(f"  PPV = {nmd_b_metrics['ppv']:.3f}, NPV = {nmd_b_metrics['npv']:.3f}")

nmd_b_specificity = nmd_b_metrics["specificity"]
nmd_b_sensitivity = nmd_b_metrics["sensitivity"]

In [ ]:

# TrunCat and TrunKitten under three threshold scenarios
binary_rows = [{"model": "NMDetective-B", "scenario": "published_>0.52",
                **nmd_b_metrics}]

for name, score_col in [("TrunCat", "score_truncat"),
                         ("TrunKitten", "score_trunkitten")]:
    scores = df[score_col].values

    # Scenario A: default 0.5
    m = bu.binary_metrics(y_true, scores, threshold=0.5)
    binary_rows.append({"model": name, "scenario": "threshold=0.5", **m})

    # Scenario B: Youden's J
    j_thr = bu.youden_threshold(y_true, scores)
    m = bu.binary_metrics(y_true, scores, threshold=j_thr)
    binary_rows.append({"model": name, "scenario": "youden_J", **m})

    # Scenario C: matched to NMDetective-B specificity
    matched_thr = bu.threshold_at_specificity(y_true, scores, nmd_b_specificity)
    m = bu.binary_metrics(y_true, scores, threshold=matched_thr)
    binary_rows.append({"model": name, "scenario": "matched_spec_to_NMDetectiveB", **m})

binary_df = pd.DataFrame(binary_rows)
binary_df.to_csv(OUTPUT_DIR / "binary_metrics.tsv", sep="\t", index=False)
print(f"[write] {OUTPUT_DIR / 'binary_metrics.tsv'}")

# Display key columns
binary_df[["model", "scenario", "threshold", "sensitivity", "specificity",
           "ppv", "npv", "balanced_accuracy", "f1"]].round(3)

## 7. Refinement analysis — within-rule-bin AUCs 
TrunCat/TrunKitten further stratify escape probability *within* each NMDetective-B rule category. Bins with too few positives or negatives are skipped.

In [ ]:

within_truncat = bu.within_bin_auc(
    df, bin_col="nmd_b_rule",
    score_col="score_truncat", label_col="y_true",
)
within_trunkitten = bu.within_bin_auc(
    df, bin_col="nmd_b_rule",
    score_col="score_trunkitten", label_col="y_true",
)
within_truncat["model"] = "TrunCat"
within_trunkitten["model"] = "TrunKitten"
within_bin = pd.concat([within_truncat, within_trunkitten], ignore_index=True)
within_bin = within_bin[["model", "bin", "n", "n_escape", "n_sensitive",
                         "roc_auc", "roc_auc_lo", "roc_auc_hi",
                         "pr_auc", "pr_auc_lo", "pr_auc_hi",
                         "skipped_reason"]]

within_bin.to_csv(OUTPUT_DIR / "within_bin_auc.tsv", sep="\t", index=False)
print(f"[write] {OUTPUT_DIR / 'within_bin_auc.tsv'}")
within_bin.round(3)

In [ ]:

# Stratification plot for TrunCat
present_bins = [b for b in RULE_ORDER if b in df["nmd_b_rule"].unique()]
fig_strat_tc, _ = bu.plot_within_bin_strip(
    df, bin_col="nmd_b_rule",
    score_col="score_truncat", label_col="y_true",
    bin_order=present_bins,
    title="TrunCat escape probability stratified by NMDetective-B rule",
    save_path=FIG_DIR / "stratification_truncat.png",
)
print(f"[write] {FIG_DIR / 'stratification_truncat.png'} (+ .pdf)")

# And for TrunKitten
fig_strat_tk, _ = bu.plot_within_bin_strip(
    df, bin_col="nmd_b_rule",
    score_col="score_trunkitten", label_col="y_true",
    bin_order=present_bins,
    title="TrunKitten escape probability stratified by NMDetective-B rule",
    save_path=FIG_DIR / "stratification_trunkitten.png",
)
print(f"[write] {FIG_DIR / 'stratification_trunkitten.png'} (+ .pdf)")

## 8. Optional — Brier score and calibration noteBrier score is reported for all three models in the performance summary(section 4). 
Note: NMDetective-B's escape score is in [0, 1] but is **notprobability-calibrated** — it's a 5-valued ranking. 
Brier comparisons shouldbe interpreted as ranking quality rather than probability quality.

In [ ]:

# Brier scores already in `summary` table; print them here for visibility
print("Brier scores (lower = better):")
for _, row in summary.iterrows():
    print(f"  {row['model']:14s}  Brier = {row['brier_score']:.4f}")

print("\nReminder: NMDetective-B is not probability-calibrated (5 discrete values).")
print("Treat its Brier as a coarse ranking-quality summary, not a calibration claim.")